# Uplift Modeling in Marketing: From *Who Will Convert* to *Who Should I Treat*

>*Application to the Hillstrom MineThatData E-Mail Analytics Challenge (2008)*

**Data Scientist:** Enio Rubens     
**Version:** v1.0       
**Dataset:** Kevin Hillstrom — MineThatData E-Mail Analytics Challenge (2008)
**Stack:** `econml`, `causalml`, `sklift`, `shap`, `mlflow`, `optpipe`

This notebook builds a **causal targeting** analysis for e-mail marketing, empirically showing why ranking customers by *propensity* (conversion probability) wastes budget and how uplift modeling identifies the customers for whom treatment truly changes the outcome.


## Notebook 1 — Framing & EDA

**Note on structure.** This project was split into multiple notebooks, one per section group, to keep each file editable. The S1-S4 history originally lived in a single notebook, now archived under `notebooks/_superseded/`. Each notebook reloads, cheaply and deterministically, what it needs from earlier sections; there is no shared kernel state across files. A cross-notebook index will be added when the split is complete. For now, the list below is a reading reference, not a fully navigable cross-file summary.

## Table of Contents (only sections 1 and 2 have functional links in this file)

1. [Framing & Setup](#s1)       
    1.1 [Technical setup](#s1-1)        
    1.2 [Context and motivation](#s1-2)     
    1.3 [The 4 quadrants of treatment response](#s1-3)      
    1.4 [Causal formalization](#s1-4)       
    1.5 [Dataset: Hillstrom MineThatData](#s1-5)        
    1.6 [Roadmap](#s1-6)        
2. [EDA & Randomization Audit](#s2)     
    2.1 [Treatment distribution](#s2-1)     
    2.2 [Univariate distributions](#s2-2)       
    2.3 [Randomization audit](#s2-3)        
    2.4 [Outcomes: distribution and raw rates](#s2-4)       
    2.5 [ATE with confidence intervals](#s2-5)      
    2.6 [Internal-validity synthesis](#s2-6)        
3. [Response-targeting baseline](02_Baseline_Propensity_EN.ipynb) — completed
4. [Meta-learners (S/T/X/R)](03_Meta_Learners_EN.ipynb) — completed
5. [Direct uplift models (Causal Forest, Uplift Trees)](04_Causal_Forest_Uplift_Trees_EN.ipynb) — completed
6. [Evaluation (Qini, AUUC, uplift@k)](05_Evaluation_Sealed_Test_EN.ipynb) — completed
7. [Heterogeneity & uplift funnel](06_Heterogeneity_Uplift_Funnel_EN.ipynb) — completed
8. [Policy learning & ROI simulation](07_Policy_Learning_ROI_EN.ipynb) — completed
9. [Robustness & limitations](08_Robustness_Limitations_EN.ipynb) — completed

---


<a id='s1'></a>
# Section 1 — Framing & Setup

**Objectives:**

1. Position the business problem (targeting under budget constraint) and the central thesis of the notebook.
2. Establish the conceptual distinction between **propensity** and **uplift** — and why this distinction matters in dollars.
3. Present the framework of the 4 quadrants of response to treatment.
4. Formalize the necessary causal concepts (ITE, ATE, CATE) without turning it into an econometrics lecture.
5. Load the Hillstrom dataset and justify the choice of primary outcome.
6. Map the next sections.

<a id='s1-1'></a>
## 1.1 Technical Setup

### 1.1.1 Development Environment

*   **Python**: 3.9 or higher
*   **libraries**: scikit-learn, pandas, numpy, matplotlib, seaborn, scikit-surprise

### 1.1.2 Dependencies

*   Install the necessary dependencies with `pip install -r requirements.txt`

### 1.1.3 Source Code

*   The source code is available in the [GitHub repository](https://github.com/usuario/repositório)

### 1.1.4 Data

*   The data used in the examples is available in the [dados.csv file](dados.csv)

### 1.1.5 Models

*

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id='s1-2'></a>
## 1.2 Context and Motivation

### The Business Problem

A retail company has an email marketing channel with **positive cost** (deliverability, fatigue of the base, opt-outs) and a **finite budget**. The classic question asked to the data team is:

> *"Which customers should I include in the next campaign?"*

The standard answer from the industry — and the one probably given in previous projects of this portfolio — is **propensity modeling**: estimate $P(\text{conversion} \mid X)$ and send an email to the top-k%. This answers the wrong question.

### The Right Question

What actually matters for the ROI of the campaign is not who will convert — it's **who will convert because of the email**. Formally: the incremental effect of the treatment per customer. In whom the email "turns the key".

This is the frontier between **predictive analytics** (what will happen) and **prescriptive/causal analytics** (what I should do). Uplift modeling lives in the second category.

### Connection with Previous Projects of this Portfolio

| Project | Question Answered | Category |
|---|---|---|
| Customer Churn Prediction | Who will cancel? | Predictive (propensity) |
| Customer Segmentation + NBA | For whom to offer what action? | Prescriptive (without causal identification) |
| Marketing Campaign Optimization | Which campaign has the best average performance? | Predictive + descriptive |
| **Uplift Modeling (this)** | **In whom the treatment makes a difference?** | **Causal** |

This notebook closes the analytical gap that was missing: causal estimation of heterogeneous effects.

<a id='s1-3'></a>
## 1.3 The 4 Quadrants of Treatment Response

For any customer $i$ and binary treatment, there are two *potential outcomes*:

- $Y_i(1)$ — what happens if the customer is treated
- $Y_i(0)$ — what happens if the customer is NOT treated

Crossing these two binary outcomes yields four customer types:

|  | $Y(0) = 0$ (does not convert without treatment) | $Y(0) = 1$ (converts without treatment) |
|---|---|---|
| **$Y(1) = 1$ (converts if treated)** | **Persuadables** — positive uplift. *The campaign ROI lives here.* | **Sure Things** — convert anyway. Treating them wastes budget. |
| **$Y(1) = 0$ (does not convert if treated)** | **Lost Causes** — never convert. Treating them is wasteful. | **Sleeping Dogs / Do Not Disturb** — **treating them reduces conversion.** They exist, and propensity modeling does not detect them. |

### Practical Implications

1. **Propensity modeling ranks by $P(Y=1 \mid X)$, which puts *Sure Things* at the top.** Budget is spent treating customers who would convert anyway.
2. **Sleeping Dogs exist in real domains** — customers who become irritated by the campaign and churn, or become more aware of alternatives. Propensity modeling treats them as good candidates. Uplift modeling identifies them as negative cost.
3. **Only *Persuadables* have positive marginal campaign ROI.** Identifying them is the goal of uplift modeling.

### Why Average ATE Hides This

If the campaign has an average conversion ATE of +2pp, this can mean:

- **Scenario A (homogeneous):** every customer has a +2pp effect. Targeting everyone is reasonable.
- **Scenario B (heterogeneous):** 10% of customers have +20pp and 90% have 0pp. Targeting only the right 10% quadruples campaign efficiency.

Both scenarios produce the **same average ATE**. Only the second allows budget optimization. Uplift modeling estimates the full distribution of individual effects, not only the mean.


<a id='s1-4'></a>
## 1.4 Causal Formalization (the Bare Minimum)

### Notation

- $T_i \in \{0, 1\}$ — treatment indicator (no email / email sent)
- $Y_i$ — observed outcome (visit / conversion / spend)
- $X_i$ — vector of pre-treatment covariates
- $Y_i(t)$ — *potential outcome* under treatment $t$

### Effects

**Individual Treatment Effect (ITE):**

$$\tau_i = Y_i(1) - Y_i(0)$$

**Fundamental Problem of Causal Inference (Holland, 1986):** $\tau_i$ is unobservable for any individual, because we never see $Y_i(0)$ and $Y_i(1)$ simultaneously.

**Average Treatment Effect (ATE):**

$$\tau = \mathbb{E}[Y(1) - Y(0)]$$

Estimable under randomization as the difference of means between arms. This is the number that standard A/B analyses report.

**Conditional Average Treatment Effect (CATE):** *This is what uplift modeling estimates.*

$$\tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]$$

### Causal Identification

Under randomization (verifiable in Section 2 via balance check), unconfoundedness is guaranteed by design:

$$\{Y(0), Y(1)\} \perp T \mid X$$

and the CATE is identifiable from observable quantities:

$$\tau(x) = \mathbb{E}[Y \mid T=1, X=x] - \mathbb{E}[Y \mid T=0, X=x]$$

The methods in Sections 4–5 (S-, T-, X-, R-learner, Causal Forest) are different strategies for estimating these two conditional expectations — each with distinct trade-offs of bias/variance.

<a id='s1-5'></a>
## 1.5 Dataset: Hillstrom MineThatData (2008)

**Origin.** Kevin Hillstrom, 2008 — challenge published on [MineThatData](http://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html). It is the canonical dataset for uplift modeling in marketing, cited in dozens of papers (Radcliffe & Surry 2011; Diemert et al. 2018; Olaya et al. 2020).

**Experimental Design.** 64,000 customers who purchased in the last 12 months, randomized into 3 arms:

- **No E-Mail** (control)
- **Mens E-Mail Campaign**
- **Womens E-Mail Campaign**

Observation window: 2 weeks post-send.

### Note

This dataset is widely used in the field of uplift modeling and has been cited in numerous research papers.

In [ ]:
from src.data import load_hillstrom

df = load_hillstrom()
print(f'Shape: {df.shape}')
print(f'\nColunas:')
for c, dt in zip(df.columns, df.dtypes):
    print(f'  {c:20s} {str(dt)}')
df.head()

### Dictionary of Variables

**Pre-processing Covariates ($X$):**

| Variable | Type | Description |
|---|---|---|
| `recency` | int | Months since last purchase (1–12) |
| `history` | float | Amount spent (USD) in the last 12 months |
| `history_segment` | cat | Historical spending range (7 levels) |
| `mens` | bin | Purchased male product in the last year |
| `womens` | bin | Purchased female product in the last year |
| `zip_code` | cat | Rural / Suburban / Urban |
| `newbie` | bin | New customer (≤12 months) |
| `channel` | cat | Phone / Web / Multichannel |

**Treatment ($T$):**

| Variable | Values |
|---|---|
| `segment` | No E-Mail / Mens E-Mail / Womens E-Mail |

**Outcomes ($Y$):**

| Variable | Type | Observed Rate |
|---|---|---|
| `visit` | bin | 14.7% — visited the site |
| `conversion` | bin | 0.9% — purchased |
| `spend` | float | zero mass, continuous between converters |

In [ ]:
from src.config import ARMS, TREATMENT_COL
from src.eda import outcome_summary_by_arm

summary = outcome_summary_by_arm(df, TREATMENT_COL, ARMS)
labels = lang({'header': 'ATE naive (Mens vs No E-Mail):'})
print(summary[['n', 'visit_rate', 'conversion_rate', 'spend_mean']])
print(f"\n{labels['header']}")
print(f'  visit:      {summary.loc["Mens E-Mail", "visit_rate"] - summary.loc["No E-Mail", "visit_rate"]:+.4f}')
print(f'  conversion: {summary.loc["Mens E-Mail", "conversion_rate"] - summary.loc["No E-Mail", "conversion_rate"]:+.4f}')
print(f'  spend:      ${summary.loc["Mens E-Mail", "spend_mean"] - summary.loc["No E-Mail", "spend_mean"]:+.4f}')

### Primary Outcome Selection

The dataset offers three outcomes, in order of proximity to the sales funnel:

1. **`visit`** (14.7% positives) — top-of-funnel engagement
2. **`conversion`** (0.9% positives) — middle/end-of-funnel
3. **`spend`** (mass of zero, continuous between converters) — monetary value

**Methodological Decision:** `visit` will be the **primary** outcome of this analysis, for the following reasons:

- **Statistics:** with 0.9% positives, `conversion` makes CATE estimation very noisy in any reasonable subgroup. Uplift modeling in ultra-rare outcomes requires samples in the order of hundreds of thousands per arm, which we do not have.
- **Literature Convention:** Radcliffe & Surry (2011), Diemert et al. (2018), and the official CausalML tutorials use `visit` as the primary outcome for the same reason.
- **Business Interpretability:** uplift in visits is a reasonable proxy for incremental engagement generated by the campaign.

`conversion` and `spend` will be treated as **secondary** outcomes in a dedicated funnel analysis (Section 7), where the divergence between uplift rankings by outcome becomes a central insight:

> *Uplift at the top of the funnel ≠ uplift at the bottom of the funnel. Optimizing for visits may not optimize for revenue.*

<a id='s1-6'></a>
## 1.6 Notebook Roadmap

| Section | Content | Output |
|---|---|---|
| **S1** | Causal framing & setup | *(this section)* |
| **S2** | EDA + randomization audit | Experimental design confirmation |
| **S3** | Baseline: response-targeting baseline | Benchmark for comparison |
| **S4** | Meta-learners (S/T/X/R) | 4 CATE estimators |
| **S5** | Direct uplift models (causal forest, UpliftTree) | Direct estimators + CIs |
| **S6** | Evaluation (Qini, AUUC, uplift@k) | Rigorous model comparison |
| **S7** | Heterogeneity analysis, quantile profiles and uplift funnel | Who are the Persuadables? |
| **S8** | Policy learning and ROI simulation | Targeting policy and economic scenarios |
| **S9** | Robustness & limitations | Sensitivity + caveats |

**Expected final output:** actionable recommendations for which clients (in terms of observable features) should be treated in the next campaign, justified with confidence intervals and ROI simulation.

<a id='s2'></a>
# Section 2 — EDA & Randomization Audit

**Objectives:**

1. Characterize the distribution of covariates and outcomes in the study population.
2. **Audit the randomization** of the experimental design — a *mandatory* step before making any causal claims.
3. Estimate naive ATEs with confidence intervals as a benchmark for later comparison with CATE estimators.
4. Diagnose the challenge of the rare outcome `conversion` that motivated the choice of `visit` as the primary outcome.

**Why this section matters.** In causal inference, *no identification, no causation*. In RCTs, identification comes from the design — randomization ensures $\{Y(0), Y(1)\} \perp T \mid X$. But "randomization" on paper is not the same as randomization in the data: implementation bugs, differential attrition, and imperfect stratification can break balance. This section verifies, with formal statistical tests, that the Hillstrom design indeed delivers what it promises.

In [ ]:
# Vetores de variáveis para uso recorrente nas próximas seções (fonte única: src/config.py)
from src.config import (
    ARMS, BIN_VARS, CAT_VARS, CONT_VARS, CONTROL_ARM, OUTCOMES,
    TREATED_ARMS, TREATMENT_COL as TREATMENT,
)

<a id='s2-1'></a>
## 2.1 Treatment Distribution

First sanity check: are the three arms comparable in size? In a 1:1:1 RCT, approximately 33% is expected in each arm. Small deviations are expected because assignment is random and not stratified by size; large deviations would indicate a design problem.


In [ ]:
from src.eda import treatment_distribution

treat_table, chi2, p = treatment_distribution(df, TREATMENT, ARMS)
labels = lang({'header': 'Chi-squared vs uniforme 1:1:1'})
print(treat_table)
print(f"\n{labels['header']}: chi2={chi2:.4f}, p={p:.4f}")

**Read-in.** Practically uniform distribution (≈33.3% per arm), and high p-value confirms that deviations are consistent with random allocation. No signature of bug in assignment.

<a id='s2-2'></a>
## 2.2 Univariate Distributions

Before the balance check proper, context about the study population.

In [ ]:
# Continuous variables
from src.viz import plot_univariate_continuous

labels = lang({
    'header': 'Variáveis contínuas',
    'dist_prefix': 'Distribuição de',
    'title': 'Distribuições univariadas — variáveis contínuas',
    'subtitle': 'history tem cauda longa; recency é aproximadamente uniforme entre 1 e 12 meses',
})
print(f"=== {labels['header']} ===")
print(df[CONT_VARS].describe().round(2))

fig, axes = plot_univariate_continuous(
    df, CONT_VARS, titles=[f"{labels['dist_prefix']} {v}" for v in CONT_VARS],
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

In [ ]:
# Categoricas e binarias
from src.viz import plot_univariate_categorical

labels = lang({
    'title': 'Distribuições univariadas — categóricas e binárias',
    'subtitle': 'mens e womens não são mutuamente excludentes — ambos ≈55%',
})
fig, axes = plot_univariate_categorical(
    df, BIN_VARS, CAT_VARS, title=labels['title'], subtitle=labels['subtitle'],
    rotate_vars=['history_segment'],
)
plt.show()

**Relevant Observations for the Model:**

- `recency` is approximately uniform between 1 and 12 months — there is no over-represented "super recent" or "super old" client.
- `history` is strongly skewed to the right (long tail of spenders). Most spent < \$200; few spent > \$1.000. Consider log transformation for linear models; trees do not need it.
- `mens` and `womens` are not mutually exclusive: both have an ≈55% rate, indicating clients who buy from both categories.
- `history_segment` is a discretization of `history` — useful as a category, but redundant for trees.

<a id='s2-3'></a>
## 2.3 Randomization audit

The heart of this section. In a successful RCT, any pre-treatment covariate should be equally distributed between arms — not exactly, but within the expected range due to random sampling.

We will use two complementary approaches:

1. **Standardized Mean Difference (SMD)** — a metric of standardized effect size from the epidemiological/econometric literature for balance. Convention:
   - $|SMD| < 0.10$ → acceptable balance
   - $|SMD| < 0.05$ → excellent balance
   - SMD is preferable to p-values because it is insensitive to sample size (with large $n$, any minimal difference becomes "significant").
2. **Formal hypothesis tests** (ANOVA F for continuous, $\chi^2$ for categorical) — to report as a secondary sanity check. We expect $p > 0.05$ for all covariates.

In [ ]:
from src.eda import build_smd_table

smd_mens = build_smd_table(df, TREATMENT, CONTROL_ARM, 'Mens E-Mail',
                            CONT_VARS, BIN_VARS, CAT_VARS)
smd_womens = build_smd_table(df, TREATMENT, CONTROL_ARM, 'Womens E-Mail',
                              CONT_VARS, BIN_VARS, CAT_VARS)

labels = lang({
    'balance_header': 'Balance: Mens E-Mail vs No E-Mail',
    'max_smd': 'Max |SMD|',
    'vars_over': 'Variáveis com |SMD| > 0.05',
    'of': 'de',
})
print(f"=== {labels['balance_header']} ===")
print(smd_mens.round(4).to_string(index=False))
print(f"\n{labels['max_smd']}: {smd_mens['smd'].abs().max():.4f}")
print(f"{labels['vars_over']}: {(smd_mens['smd'].abs() > 0.05).sum()} {labels['of']} {len(smd_mens)}")

In [ ]:
labels = lang({
    'balance_header': 'Balance: Womens E-Mail vs No E-Mail',
    'max_smd': 'Max |SMD|',
    'vars_over': 'Variáveis com |SMD| > 0.05',
    'of': 'de',
})
print(f"=== {labels['balance_header']} ===")
print(smd_womens.round(4).to_string(index=False))
print(f"\n{labels['max_smd']}: {smd_womens['smd'].abs().max():.4f}")
print(f"{labels['vars_over']}: {(smd_womens['smd'].abs() > 0.05).sum()} {labels['of']} {len(smd_womens)}")

In [ ]:
from src.viz import plot_love_plot

labels = lang({
    'title': 'Balance covariáveis pré-tratamento',
    'subtitle': 'Todos os |SMD| abaixo de 0.017 — balance excelente nos dois braços',
})
fig, ax = plot_love_plot(
    {'Mens E-Mail vs Control': smd_mens, 'Womens E-Mail vs Control': smd_womens},
    title=labels['title'],
    xlabel='Standardized Mean Difference',
    subtitle=labels['subtitle'],
)
plt.show()

In [ ]:
from src.eda import formal_balance_tests

formal_tests = formal_balance_tests(df, TREATMENT, ARMS, CONT_VARS, BIN_VARS + CAT_VARS)
labels = lang({
    'header': 'Testes de balance (3-way: No / Mens / Womens)',
    'continuous': 'Contínuas (ANOVA F)',
    'categorical': 'Categóricas e binárias (chi-squared)',
})
print(f"=== {labels['header']} ===\n")
print(f"-- {labels['continuous']} --")
print(formal_tests[formal_tests['test'] == 'ANOVA F'].to_string(index=False))
print(f"\n-- {labels['categorical']} --")
print(formal_tests[formal_tests['test'] == 'chi-squared'].to_string(index=False))

**Conclusion of the randomization audit.**

- **All SMDs remained well below 0.05** in both comparisons (Mens vs Control, Womens vs Control). The maximum observed is of the order of 0.01 — *an order of magnitude below the excellence threshold.*
- **No formal test rejects H₀** of equal distribution (all with $p > 0.3$).
- The experimental design is, therefore, **internally valid**. The estimated ATEs that follow can be interpreted causally without the need for adjustment for observable confounding.

> Compare with the dataset from the previous project (Marketing Campaign Optimization), where `language_preferred` diverged ~5pp between arms and `marketing_channel` ~6pp — requiring adjustments for propensity. Here, this work is not necessary.

<a id='s2-4'></a>
## 2.4 Outcomes: Distribution and Gross Rates

Before estimating ATEs with CI, a panorama of outcomes by arm.

In [ ]:
from src.eda import outcome_summary_by_arm

outcomes_by_arm = outcome_summary_by_arm(df, TREATMENT, ARMS)
print(outcomes_by_arm)

In [ ]:
from src.viz import plot_outcomes_by_arm

labels = lang({
    'visit_rate': 'Visit rate',
    'conversion_rate': 'Conversion rate',
    'spend_mean': 'Spend médio ($)',
    'per_arm_suffix': 'por braço',
    'title': 'Outcomes por braço',
    'subtitle': 'Mens E-Mail lidera nos três outcomes; Womens vem em segundo',
})
fig, axes = plot_outcomes_by_arm(
    df, TREATMENT, ARMS,
    outcomes=['visit', 'conversion', 'spend'],
    ylabels=[labels['visit_rate'], labels['conversion_rate'], labels['spend_mean']],
    titles=[
        f"{labels['visit_rate']} {labels['per_arm_suffix']}",
        f"{labels['conversion_rate']} {labels['per_arm_suffix']}",
        f"{labels['spend_mean']} {labels['per_arm_suffix']}",
    ],
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

### Diagnosis of outcome `conversion` (rare class)

Note: only **578 conversions in 64,000 observations** (~0.9%). Between the arms:

- In Email: ~122 conversions in 21,306 → 0.57%
- Men's Email: ~267 conversions in 21,307 → 1.25%
- Women's Email: ~189 conversions in 21,387 → 0.88%

To estimate CATE in relevant subgroups (e.g., customers with `recency <= 3`), the event sizes per subgroup × arm drop to dozens of conversions. This is insufficient for stable uplift modeling in `conversion` — the variance of the estimates explodes.

This is the empirical justification for using `visit` (9,394 total events, ~14.7%) as the primary outcome.

<a id='s2-5'></a>
## 2.5 ATE with Confidence Intervals

ATEs estimated as simple difference of means between arms. This is justified by randomization (S2.3); it would be insufficient under observational design.

**Inference:**
- Binary outcomes (`visit`, `conversion`): Normal CI with standard error of the difference in proportions.
- Continuous outcome (`spend`): Welch t-test (does not assume equal variances).

### Note
The use of confidence intervals for ATEs is a common practice in A/B testing. However, it's worth noting that the choice of interval type and calculation method may depend on the specific characteristics of the data and the research question at hand.

In [ ]:
from src.effects import ate_table

ate_results = ate_table(
    df, TREATMENT, TREATED_ARMS, CONTROL_ARM,
    binary_outcomes=['visit', 'conversion'], continuous_outcome='spend',
)
print(ate_results.to_string(index=False))

In [ ]:
from src.viz import plot_ate_forest

labels = lang({
    'title': 'Efeito do tratamento (ATE) por outcome',
    'subtitle': 'Efeito positivo e significativo (p < 0.001) em ambos os braços, nos três outcomes',
})
fig, axes = plot_ate_forest(
    ate_results, outcomes=['visit', 'conversion', 'spend'],
    xlabel_prefix='ATE: ', title_prefix='Effect of treatment on ',
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

**Reading ATEs.**

In **all** outcomes and **both** treatments, the effect is statistically significant ($p < 0.001$) and positive:

- **Visit (top of funnel):** Mens E-Mail pushes the rate by +7.66pp; Womens E-Mail by +4.52pp. The difference between the two treatments is clearly significant (CIs do not overlap).
- **Conversion (bottom of funnel):** Mens +0.68pp, Womens +0.31pp. Although the size is small, the separation of CIs suggests that Mens E-Mail also dominates here.
- **Spend:** Mens E-Mail generates ~\$0.77/client incremental revenue; Womens ~\$0.42. On a base of 1M clients, this is the difference between \$770K and \$420K in incremental revenue — not negligible.

**Observation heterogeneity.** The fact that Mens dominates Womens *on average* does not mean that Mens dominates Womens *for every client*. It is plausible — even likely — that there are subgroups (e.g., clients who only buy `womens`) where Womens E-Mail is more effective. This is exactly the hypothesis that justifies uplift modeling: average ATEs hide exploratory heterogeneity.

<a id='s2-6'></a>
## 2.6 Synthesis: Causal Identification is Guaranteed

This section established three necessary things before proceeding to the estimation of CATE:

| Question | Response | Evidence |
|---|---|---|
| Do the arms have comparable sizes? | Yes, ~33% each | $\chi^2$ against uniform does not reject |
| Are the covariates balanced? | Yes, at an excellent level | All \|SMD\| < 0.015; all formal tests with $p > 0.3$ |
| Is there a signal of treatment? | Yes, on all outcomes | ATEs all significant at $p < 0.001$ |

**Practical Consequence for the Next Sections:**
- *Unconfoundedness* is guaranteed by design — no propensity adjustment necessary.
- The CATE is identified as $\tau(x) = E[Y \mid T=1, X=x] - E[Y \mid T=0, X=x]$ without bias.
- S4 (S/T/X/R-learner) and S5 (Causal Forest, UpliftTree) methods are estimating a real causal quantity, not a confounded artifact.

**Next:** [Section 3 — response-targeting baseline model](#s3) (benchmark for comparison with uplift).